# Simple: Text Semantic Categorization with PAMOLA.CORE

**Goal**: Learn text semantic categorization basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Apply semantic categorization using execute()
- Compare before/after results
- Understand entity extraction and categorization

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/analyzers/
        └── 01_text_semantic_categorizer_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.text import (
        TextSemanticCategorizerOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load data from `examples/data_examples/sample.csv`
- Auto-creates sample data if file missing
- Review dataset structure before proceeding

**What you'll see:**
- Dataset summary (20 records, 3 columns)
- First 5 records with job titles
- Column statistics showing unique values
- Text fields: job_title (20 unique values), department, experience_years

**Note:** Sample includes job titles suitable for semantic categorization (roles, seniority levels)

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with job titles for entity extraction
    sample_data = pd.DataFrame({
        'record_id': range(1, 21),
        'job_title': [
            'Senior Software Engineer', 'Data Scientist', 'Product Manager',
            'Machine Learning Engineer', 'DevOps Engineer', 'Full Stack Developer',
            'Backend Developer', 'Frontend Developer', 'Data Analyst',
            'UX Designer', 'Senior Data Engineer', 'Cloud Architect',
            'Software Developer', 'AI Research Scientist', 'Technical Lead',
            'Junior Developer', 'Senior Product Manager', 'Business Analyst',
            'System Administrator', 'Database Administrator'
        ],
        'department': [
            'Engineering', 'Data Science', 'Product', 'Data Science', 'Engineering',
            'Engineering', 'Engineering', 'Engineering', 'Analytics',
            'Design', 'Engineering', 'Engineering', 'Engineering',
            'Research', 'Engineering', 'Engineering', 'Product',
            'Analytics', 'IT', 'IT'
        ],
        'experience_years': [
            5, 3, 4, 4, 6, 3, 2, 2, 2,
            3, 7, 8, 1, 5, 10, 0, 6, 3, 4, 5
        ]
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE field_name** in `create_config_kwargs()`
   - Default: `"job_title_current"`
   - Change to YOUR dataset's text column name
2. Optionally customize `id_field` for record identification
3. Run to validate fields and setup environment

**What you'll see:**
- Field validation (unique values, sample values, text length stats)
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and progress tracker ready (✓)

**Note:** Field must exist in dataset and contain text strings suitable for categorization

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "job_title_current",  # Field to process - CUSTOMIZE THIS!
        "id_field": "resume_id"
    }

kwargs = create_config_kwargs()
field_name = kwargs["field_name"]
id_field = kwargs["id_field"]

# Validate field exists
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

print(f"✓ Field to process: '{field_name}'")
print(f"  Unique values: {df[field_name].nunique()}")
print(f"  Sample values: {list(df[field_name].unique()[:5])}")
print(f"  Text length stats: min={df[field_name].str.len().min()}, max={df[field_name].str.len().max()}, mean={df[field_name].str.len().mean():.1f}")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="text_semantic_categorization",
    description="Semantic categorization of text field using entity extraction.",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Processing {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review configuration parameters below
- Run to execute semantic categorization
- Monitor progress bar (6 steps: load → analyze → match → cluster → categorize → complete)

**Key parameters:**
- `field_name`: Field to analyze (text required)
- `entity_type='job'`: Type of entities to extract (job, location, skill)
- `perform_categorization=True`: Enable semantic categorization
- `use_ner=True`: Named Entity Recognition for unmatched texts
- `perform_clustering=True`: Cluster similar items by semantic similarity

**What you'll see:**
- Configuration summary with all parameters
- Progress bar: text analysis → dictionary matching → NER extraction → clustering → save → complete
- Completion status: "completed" (verify no errors)
- Artifact count (6-8 files expected)

**Note:** Uses multi-strategy matching: dictionary → NER → clustering for comprehensive categorization

In [ ]:
# Create operation with production-style configuration
operation = TextSemanticCategorizerOperation(
    # Core parameters
    field_name=field_name,
    id_field=id_field,
    entity_type='job',
    
    # Dictionary configuration
    dictionary_path=None,  # Will auto-detect from repository
    
    # Text processing parameters
    min_word_length=3,
    
    # Categorization parameters
    perform_categorization=True,
    match_strategy='specific_first',
    use_ner=True,
    
    # Clustering parameters
    perform_clustering=True,
    clustering_threshold=0.7,
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Field:            {operation.field_name}")
print(f"  Entity type:      {operation.entity_type}")
print(f"  Categorization:   {operation.perform_categorization}")
print(f"  Use NER:          {operation.use_ner}")
print(f"  Clustering:       {operation.perform_clustering}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing text semantic categorization...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load categorization results from task_dir
- Review match summary and category distributions
- Verify categorization quality

**What you'll see:**
- Text quality metrics (null %, empty %, avg length, language)
- Match summary (dictionary matched, NER matched, clustered, unresolved)
- Category distribution (top 5 categories with counts)
- Alias distribution (top 5 text variations)
- Match percentage showing categorization success rate

**Note:** High match percentage (>80%) indicates good dictionary coverage and entity recognition

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file (JSON with analysis results)
output_files = list(task_dir.glob('output/*.json'))
if output_files:
    output_file = output_files[0]
    
    with open(output_file, 'r') as f:
        analysis_results = json.load(f)
    
    print("\n" + "=" * 80)
    print("📊 RESULTS ANALYSIS")
    print("=" * 80)
    
    # Show basic text analysis
    if 'null_empty_analysis' in analysis_results:
        null_analysis = analysis_results['null_empty_analysis']
        print("\n📝 Text Quality:")
        print(f"  Total records:        {null_analysis['total_records']}")
        print(f"  Actual data:          {null_analysis['actual_data']['count']} ({null_analysis['actual_data']['percentage']:.1f}%)")
        print(f"  Null values:          {null_analysis['null_values']['count']} ({null_analysis['null_values']['percentage']:.1f}%)")
        print(f"  Empty strings:        {null_analysis['empty_strings']['count']} ({null_analysis['empty_strings']['percentage']:.1f}%)")
    
    # Show language analysis
    if 'language_analysis' in analysis_results:
        lang_analysis = analysis_results['language_analysis']
        print(f"\n🌐 Language Analysis:")
        print(f"  Predominant language: {lang_analysis['predominant_language']}")
    
    # Show length statistics
    if 'length_stats' in analysis_results:
        length_stats = analysis_results['length_stats']
        print(f"\n📏 Text Length Statistics:")
        print(f"  Min length:           {length_stats['min']}")
        print(f"  Max length:           {length_stats['max']}")
        print(f"  Mean length:          {length_stats['mean']:.1f}")
        print(f"  Median length:        {length_stats['median']}")
    
    # Show categorization summary
    if 'match_summary' in analysis_results:
        summary = analysis_results['match_summary']
        print("\n" + "=" * 80)
        print("✨ CATEGORIZATION SUMMARY")
        print("=" * 80)
        print(f"  Total texts:          {summary['total_texts']}")
        print(f"  Dictionary matched:   {summary['num_matched']} ({summary['percentage_matched']:.1f}%)")
        print(f"  NER matched:          {summary['num_ner_matched']}")
        print(f"  Clustered:            {summary['num_auto_clustered']}")
        print(f"  Unresolved:           {summary['num_unresolved']}")
    
    # Show top categories
    if 'category_distribution' in analysis_results:
        print(f"\n📊 Top Categories:")
        cat_dist = analysis_results['category_distribution']
        for i, (category, count) in enumerate(list(cat_dist.items())[:5], 1):
            print(f"  {i}. {category:30s} {count:3d} texts")
    
    # Show top aliases
    if 'aliases_distribution' in analysis_results:
        print(f"\n🔖 Top Aliases:")
        alias_dist = analysis_results['aliases_distribution']
        for i, (alias, count) in enumerate(list(alias_dist.items())[:5], 1):
            print(f"  {i}. {alias:30s} {count:3d} texts")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Semantic categorization helps understand text patterns and entities!")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Run to display all generated files
- Navigate to directories for manual inspection
- Verify each folder has expected file count

**What you'll see:**
- Directory structure tree (output/, dictionaries/, metrics/, visualizations/, config/)
- File count per directory (typically 2-4 files each)
- File names with sizes in KB
- Absolute path to task directory for manual navigation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Analysis results JSON
├── dictionaries/     # Semantic roles, category mappings, unresolved terms CSVs
├── metrics/          # Performance metrics JSON
├── visualizations/   # Distribution charts (PNG)
└── config/           # Operation configuration
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'dictionaries', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Text quality and categorization effectiveness metrics
2. **Analysis Results**: Detailed categorization with match methods and confidence
3. **Category Mappings**: CSV showing original text → category mappings with domains
4. **Visualizations**: Distribution charts for categories and match methods

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types files
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]
        
        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # Display metadata
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")
            if 'operation' in metadata:
                op = metadata['operation']
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")
            
            # Display text quality metrics
            print("\n📝 TEXT QUALITY:")
            print(f"   Total Records: {metrics.get('total_records', 'N/A')}")
            print(f"   Null %: {metrics.get('null_percentage', 0):.2f}%")
            print(f"   Empty %: {metrics.get('empty_percentage', 0):.2f}%")
            print(f"   Avg Text Length: {metrics.get('avg_text_length', 0):.1f}")
            print(f"   Max Text Length: {metrics.get('max_text_length', 0)}")
            print(f"   Predominant Language: {metrics.get('predominant_language', 'N/A')}")
            
            # Display categorization metrics
            if 'num_matched' in metrics:
                print("\n🔍 CATEGORIZATION:")
                print(f"   Dictionary Matched: {metrics.get('num_matched', 0)}")
                print(f"   NER Matched: {metrics.get('num_ner_matched', 0)}")
                print(f"   Auto Clustered: {metrics.get('num_auto_clustered', 0)}")
                print(f"   Unresolved: {metrics.get('num_unresolved', 0)}")
                print(f"   Match Rate: {metrics.get('percentage_matched', 0):.2f}%")
            
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. ANALYSIS RESULTS (NEWEST)
print("\n\n2️⃣ ANALYSIS RESULTS:")
print("-" * 80)

output_dir = task_dir / "output"
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob("*.json"))

    # 👉 keep ONLY categorization analysis
    output_files = [
        f for f in output_files
        if "stats" not in f.name
        and "dictionary" not in f.name
        and "phone" not in f.name
    ]

    if not output_files:
        print("⚠️  No categorization analysis file found")
    else:
        output_files.sort(key=lambda f: f.stat().st_mtime, reverse=True)
        analysis_file = output_files[0]

        mtime = datetime.fromtimestamp(analysis_file.stat().st_mtime)
        print(f"📄 Loading LATEST: {analysis_file.name}")
        print(f"🕒 Modified: {mtime:%Y-%m-%d %H:%M:%S}")
        print(f"📦 Size: {analysis_file.stat().st_size / 1024:.1f} KB\n")
        try:
            with open(analysis_file, "r", encoding="utf-8") as f:
                data = json.load(f)

            # ==================================================
            # BASIC INFO
            # ==================================================
            print("📌 BASIC INFO")
            print(f"  Field name : {data.get('field_name')}")
            print(f"  Entity     : {data.get('entity_type')}")
            print()

            # ==================================================
            # NULL / EMPTY ANALYSIS
            # ==================================================
            ne = data.get("null_empty_analysis", {})
            print("🧹 NULL / EMPTY ANALYSIS")
            print(f"  Total records : {ne.get('total_records', 0):,}")
            print(f"  Null          : {ne.get('null_values', {}).get('percentage', 0):.2f}%")
            print(f"  Empty string  : {ne.get('empty_strings', {}).get('percentage', 0):.2f}%")
            print(f"  Actual data   : {ne.get('actual_data', {}).get('percentage', 0):.2f}%")
            print()

            # ==================================================
            # LENGTH STATISTICS
            # ==================================================
            ls = data.get("length_stats", {})
            print("📏 TEXT LENGTH STATS")
            print(f"  Min / Max : {ls.get('min')} / {ls.get('max')}")
            print(f"  Mean      : {ls.get('mean'):.2f}")
            print(f"  Median    : {ls.get('median')}")
            print(f"  Std       : {ls.get('std'):.2f}")
            print()

            # ==================================================
            # LANGUAGE
            # ==================================================
            lang = data.get("language_analysis", {})
            print("🌐 LANGUAGE")
            print(f"  Predominant language : {lang.get('predominant_language')}")
            print()

            # ==================================================
            # MATCH SUMMARY
            # ==================================================
            ms = data.get("match_summary", {})
            print("🎯 MATCH SUMMARY")
            print(f"  Total texts        : {ms.get('total_texts', 0):,}")
            print(f"  Matched            : {ms.get('num_matched', 0):,}")
            print(f"  Auto-clustered     : {ms.get('num_auto_clustered', 0):,}")
            print(f"  Unresolved         : {ms.get('num_unresolved', 0):,}")
            print(f"  Match percentage   : {ms.get('percentage_matched', 0):.2f}%")

            if ms.get("top_categories"):
                print("\n  Top categories:")
                for c in ms["top_categories"][:5]:
                    print(f"    • {c}")
            print()

            # ==================================================
            # CATEGORY DISTRIBUTION (TOP 10)
            # ==================================================
            print("📊 CATEGORY DISTRIBUTION (TOP 10)")
            cat_dist = data.get("category_distribution", {})
            for cat, cnt in sorted(cat_dist.items(), key=lambda x: x[1], reverse=True)[:10]:
                print(f"  {cat:30s} {cnt:5d}")
            print()

            # ==================================================
            # SAMPLE CATEGORIZATION
            # ==================================================
            print("🧪 SAMPLE CATEGORIZATIONS")
            for item in data.get("categorization", [])[:5]:
                print(
                    f"  • {item.get('original_text','')[:30]:30s} → "
                    f"{item.get('matched_category','N/A'):25s} "
                    f"(method={item.get('match_method')}, "
                    f"confidence={item.get('match_confidence', item.get('score','-'))})"
                )

            print("\n" + "=" * 80)
            print("✅ JOB TITLE ANALYSIS COMPLETE")
            
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 3. CATEGORY MAPPINGS (NEWEST)
print("\n\n3️⃣ CATEGORY MAPPINGS:")
print("-" * 80)

dict_dir = task_dir / "dictionaries"
if not dict_dir.exists():
    print(f"⚠️  Dictionaries directory not found: {dict_dir}")
else:
    mapping_files = list(dict_dir.glob("*_category_mappings_*.csv"))

    if not mapping_files:
        print("⚠️  No mapping files found")
    else:
        mapping_files.sort(key=lambda f: f.stat().st_mtime, reverse=True)
        latest_mapping = mapping_files[0]

        mtime = datetime.fromtimestamp(latest_mapping.stat().st_mtime)
        print(f"📄 Loading LATEST: {latest_mapping.name}")
        print(f"🕒 Modified: {mtime:%Y-%m-%d %H:%M:%S}")
        print(f"📦 Size: {latest_mapping.stat().st_size / 1024:.1f} KB\n")

        try:
            mapping_df = pd.read_csv(latest_mapping)

            # ==================================================
            # BASIC STATS
            # ==================================================
            print("📊 BASIC STATS")
            print(f"  Rows        : {len(mapping_df):,}")
            print(f"  Columns     : {len(mapping_df.columns)}")
            print(f"  Record IDs  : {mapping_df['record_id'].nunique():,}")
            print(f"  Job titles  : {mapping_df['original_text'].nunique():,}")
            print()

            # ==================================================
            # DEDUPLICATION CHECK
            # ==================================================
            dup_mask = mapping_df.duplicated(
                subset=["record_id", "original_text", "matched_category"]
            )
            dup_count = dup_mask.sum()

            print("🧹 DUPLICATION")
            print(f"  Duplicate rows detected : {dup_count:,}")
            if dup_count > 0:
                print(f"  Effective unique rows   : {len(mapping_df) - dup_count:,}")
            print()

            # ==================================================
            # MATCH METHOD DISTRIBUTION
            # ==================================================
            print("🎯 MATCH METHOD DISTRIBUTION")
            if "match_method" in mapping_df.columns:
                method_dist = mapping_df["match_method"].value_counts()
                for method, cnt in method_dist.items():
                    pct = cnt / len(mapping_df) * 100
                    print(f"  {method:15s}: {cnt:4d} ({pct:5.1f}%)")
            print()

            # ==================================================
            # DOMAIN DISTRIBUTION
            # ==================================================
            print("🏷️ DOMAIN DISTRIBUTION")
            domain_dist = mapping_df["matched_domain"].value_counts()
            for domain, cnt in domain_dist.items():
                pct = cnt / len(mapping_df) * 100
                print(f"  {domain:25s}: {cnt:4d} ({pct:5.1f}%)")
            print()

            # ==================================================
            # TOP CLUSTERS
            # ==================================================
            cluster_df = mapping_df[mapping_df["matched_domain"] == "Cluster"]

            if not cluster_df.empty:
                print("🧩 TOP CLUSTERS (by frequency)")
                top_clusters = (
                    cluster_df["matched_category"]
                    .value_counts()
                    .head(10)
                )
                for cat, cnt in top_clusters.items():
                    print(f"  {cat:25s}: {cnt:4d}")
                print()

            # ==================================================
            # SUSPICIOUS MAPPINGS (HEURISTIC)
            # ==================================================
            print("⚠️ SUSPICIOUS MAPPINGS (Heuristic)")
            suspicious = mapping_df[
                (
                    mapping_df["original_text"].str.contains(
                        "SRE|QA|DevOps", case=False, na=False
                    )
                )
                &
                (
                    mapping_df["matched_category"].str.contains(
                        "Frontend|Data_Engineer", case=False, na=False
                    )
                )
            ]

            if suspicious.empty:
                print("  ✓ No obvious suspicious mappings found")
            else:
                print(f"  Found {len(suspicious)} suspicious rows (sample):")
                print(
                    suspicious[
                        ["record_id", "original_text", "matched_category", "matched_domain"]
                    ]
                    .head(10)
                    .to_string(index=False)
                )

            print("\n" + "=" * 80)
            print("✅ CATEGORY MAPPING ANALYSIS COMPLETE")
        except Exception as e: 
            print(f"❌ Error reading mapping: {e}")

# 4. VISUALIZATIONS (NEWEST BATCH)
print("\n\n4️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            time_threshold = 10
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            latest_viz_batch.sort(key=lambda x: x.name)
            
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            for i, viz_file in enumerate(latest_viz_batch, 1):
                viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**  
✅ Load data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure text semantic categorization with full parameters  
✅ Execute using operation.execute()  
✅ Analyze results and review artifacts  

**Key takeaways:**
- Semantic categorization extracts entities and patterns from text
- Multiple matching strategies: dictionary, NER, clustering
- Language detection helps optimize processing
- Category hierarchies enable structured analysis
- Unresolved terms can be clustered by similarity

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_text_semantic_categorizer_advanced.ipynb`** includes:
- Custom dictionary creation and management
- Advanced entity extraction strategies
- Multi-language support
- Hierarchical categorization
- Performance optimization for large datasets
- Integration with other PAMOLA operations

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 📚 [NLP Utilities Guide](../../docs/nlp_utilities.md)